# Basic Data Inspection

This notebook checks whether the Raisin Dataset is ready for modeling. Each section answers one required data-quality question.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

sns.set_theme(
    style="whitegrid",
    context="notebook",
)

RANDOM_SEED = 42

## 1. Dataset Loading

The unchanged Raisin Dataset is loaded from the project's raw-data directory.

In [ ]:
CURRENT_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name == "notebooks"
    else CURRENT_DIRECTORY
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "raisin"
    / "Raisin_Dataset.xlsx"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_excel(DATA_PATH)
df.head()


## 2. Dataset Dimensions and Column Names

This section reports the number of observations, variables, and available column names.

In [ ]:
number_of_rows, number_of_columns = df.shape

print(f"Number of rows: {number_of_rows:,}")
print(f"Number of columns: {number_of_columns}")

for column_number, column_name in enumerate(df.columns, start=1):
    print(f"{column_number}. {column_name}")

## 3. Data Types

This section examines the data type, non-null count, and memory usage of each column.

In [ ]:
df.info()

In [ ]:
data_type_summary = pd.DataFrame(
    {
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.notna().sum().values,
        "Unique Values": df.nunique().values,
    }
)

data_type_summary

## 4. Target-Class Distribution

The target column is inspected to verify the number of classes, class frequencies, and class balance.

In [ ]:
TARGET_COLUMN = "Class"

number_of_target_classes = df[TARGET_COLUMN].nunique()

print(f"Target column: {TARGET_COLUMN}")
print(f"Number of target classes: {number_of_target_classes}")

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()

target_percentages = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .mul(100)
)

target_summary = pd.DataFrame(
    {
        "Count": target_counts,
        "Percentage": target_percentages,
    }
)

target_summary.index.name = "Class"

target_summary

## 5. Missing-Value Analysis

This section calculates the number and percentage of missing values in each column.

In [ ]:
missing_count = df.isna().sum()

missing_percentage = (
    df.isna()
    .mean()
    .mul(100)
)

missing_summary = pd.DataFrame(
    {
        "Missing Count": missing_count,
        "Missing Percentage": missing_percentage,
    }
)

missing_summary

In [ ]:
total_missing_values = df.isna().sum().sum()

print(f"Total missing values: {total_missing_values}")

## 6. Duplicate-Row Analysis

Duplicate observations are checked because identical records could bias the analysis or appear in different data splits.

In [ ]:
duplicate_count = df.duplicated().sum()

duplicate_percentage = (
    duplicate_count / len(df)
) * 100

print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

In [ ]:
if duplicate_count > 0:
    duplicate_rows = df[df.duplicated(keep=False)]
    display(duplicate_rows)
else:
    print("No duplicate rows were found.")

## 7. Descriptive Statistics

Summary statistics are calculated for all numerical input features.

In [ ]:
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

In [ ]:
descriptive_statistics = (
    df[numerical_columns]
    .describe()
    .transpose()
)

descriptive_statistics

The table includes:
- count: number of available observations
- mean: arithmetic average
- std: standard deviation
- min: minimum value
- 25%: first quartile
- 50%: median
- 75%: third quartile
- max: maximum value

In [ ]:
descriptive_statistics["range"] = (
    descriptive_statistics["max"]
    - descriptive_statistics["min"]
)

descriptive_statistics["IQR"] = (
    descriptive_statistics["75%"]
    - descriptive_statistics["25%"]
)

descriptive_statistics

## 8. Categorical-Value Frequencies

Categorical columns are identified, and their unique values and frequencies are examined.

In [ ]:
categorical_columns = df.select_dtypes(
    include=["str", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

## 9. Identifier, Timestamp, and Leakage Audit

Columns are inspected for identifiers, timestamps, direct target copies, and information that would not be available before prediction.

In [ ]:
identifier_keywords = {
    "id",
    "identifier",
    "index",
    "sample_id",
    "record_id",
    "code",
}

timestamp_keywords = {
    "date",
    "time",
    "timestamp",
    "created_at",
    "updated_at",
}

identifier_candidates = [
    column_name
    for column_name in df.columns
    if column_name.lower() in identifier_keywords
    or column_name.lower().endswith("_id")
]

timestamp_candidates = [
    column_name
    for column_name in df.columns
    if column_name.lower() in timestamp_keywords
    or "date" in column_name.lower()
    or "timestamp" in column_name.lower()
]

print("Possible identifier columns:", identifier_candidates)
print("Possible timestamp columns:", timestamp_candidates)

In [ ]:
input_columns = [
    column_name
    for column_name in df.columns
    if column_name != TARGET_COLUMN
]

print("Input columns:")
print(input_columns)

print("\nTarget column:")
print(TARGET_COLUMN)

### Leakage Conclusion

No identifier, timestamp, target copy, or post-classification feature was found. The seven inputs are physical measurements available before classification, so no obvious target leakage is present.